# Module 5 — Portfolio Construction

## Overview

M4 identified **18 STRONG factors** across three implementation
tiers. M5 translates these signal quality ratings into actual
portfolio weights using three complementary construction methods:

| Portfolio | Method | Purpose |
|-----------|--------|---------|
| **Equal-weight (EW)** | $w_j = 1/N$ for all STRONG factors | Benchmark - no optimisation |
| **Mean-Variance (MVO)** | Maximise risk-adjusted return with turnover penalty | Primary - optimised |
| **Risk-Parity (RP)** | Weight by inverse factor volatility | Robustness - no return estimates |

The EW benchmark answers: how much does optimisation add over
simply holding all strong factors equally?
The MVO portfolio is our primary result.
The RP portfolio is robust to expected return estimation error.

## From Factor Portfolios to Stock Portfolios

Each of the 18 STRONG factors is already a long-short portfolio
of stocks (constructed in M1). The M5 portfolio is a
**portfolio of factor portfolios** - a linear combination of
the 18 long-short factor returns:

$$R_{p,t} = \sum_{j=1}^{J} w_j F_{j,t}$$

where $w_j$ is the weight on factor $j$ and $F_{j,t}$ is its
month-$t$ long-short return from M1.

## Mean-Variance Optimisation

The MVO portfolio solves:

$$\max_{\mathbf{w}} \; \mathbf{w}^\top \boldsymbol{\mu}
  - \frac{\gamma}{2} \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}
  - \lambda \|\mathbf{w} - \mathbf{w}_{t-1}\|_1$$

where:
- $\boldsymbol{\mu}$ = expected factor returns (from M4 signal scores)
- $\boldsymbol{\Sigma}$ = factor return covariance matrix (from M1)
- $\gamma$ = risk aversion parameter (set to 2.0)
- $\lambda$ = turnover penalty (scaled by M4 implementation tier)

**Turnover penalty by tier:**
- Tier 1 (STRONG/HIGH): $\lambda = 0.001$ -- low penalty, slow-moving
- Tier 2 (STRONG/MEDIUM): $\lambda = 0.005$ -- moderate penalty
- Tier 3 (STRONG/LOW): $\lambda = 0.020$ -- heavy penalty, fast-moving

**Constraints:**
- $\sum_j w_j = 1$ (fully invested in factor space)
- $w_j \geq 0$ (long-only factor weights — each factor is already
  long-short at the stock level)
- $w_j \leq 0.30$ (maximum 30% in any single factor)

## Inputs and Outputs

**Inputs:**
- `outputs/M1/factor_returns_41.parquet` — factor return series
- `outputs/M4/signal_dashboard.csv` — signal scores and tiers
- `outputs/M4/ic_results.csv` — ICIR for expected return proxy

**Outputs** (saved to `outputs/M5/`):
- `portfolio_weights.csv` — static weights for all three portfolios
- `portfolio_returns.parquet` — monthly returns for all three
- `portfolio_stats.csv` — Sharpe, drawdown, turnover statistics
- `figures/` — cumulative return plots, weight charts

In [3]:
# Imports and Configuration

import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.io as pio
import os, warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook"

# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = (r"C:\Users\amosa\Documents\My Graduate School"
                r"\SPRING 2026\Courses\AMS520_Machine Learning in Finance"
                r"\Project\General ML4Trading Resource"
                r"\Personal_Fundamental Factor Investing"
                r"\fundamental-factor-investing")
M1_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M1")
M4_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M4")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "M5")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Factor families ────────────────────────────────────────────
CHAR_FAMILIES = {
    "Value":         ["BEME","E2P","CF2P","D2P","S2P","A2ME"],
    "Profitability": ["PROF","ROE","ROA","OP","PM","PCM","RNA"],
    "Investment":    ["Investment","NOA","DPI2A","NI","OA","AC"],
    "Momentum":      ["r12_2","r2_1","r12_7","r36_13",
                      "LT_Rev","SUV","Rel2High"],
    "Risk":          ["Beta","IdioVol","Variance",
                      "Spread","LTurnover","LME"],
    "Other":         ["Q","C","AT","ATO","CTO",
                      "D2A","FC2Y","Lev","SGA2S"],
}
ALL_41 = [c for fam in CHAR_FAMILIES.values() for c in fam]

FAMILY_COLORS = {
    "Value":         "#1f77b4",
    "Profitability": "#2ca02c",
    "Investment":    "#ff7f0e",
    "Momentum":      "#d62728",
    "Risk":          "#9467bd",
    "Other":         "#8c564b",
}

# ── MVO parameters ────────────────────────────────────────────
GAMMA       = 2.0    # risk aversion
MAX_WEIGHT  = 0.30   # maximum weight per factor

# Turnover penalty by implementation tier
LAMBDA_BY_TIER = {
    'Tier 1': 0.001,   # STRONG/HIGH  — slow-moving
    'Tier 2': 0.005,   # STRONG/MEDIUM — moderate
    'Tier 3': 0.020,   # STRONG/LOW   — fast-moving
}

print(f"Output directory: {OUT_DIR}")
print(f"MVO parameters:   γ={GAMMA}, max_weight={MAX_WEIGHT:.0%}")

Output directory: C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Investing\fundamental-factor-investing\outputs\M5
MVO parameters:   γ=2.0, max_weight=30%


In [4]:
# Load data and define investment universe

print("Loading data...")

# Factor returns from M1
factor_returns = pd.read_parquet(
    os.path.join(M1_OUT, "factor_returns_41.parquet"))
factor_returns.index = (pd.to_datetime(factor_returns.index)
                        + pd.offsets.MonthEnd(0))
factor_returns = factor_returns[ALL_41].copy()

# Signal dashboard from M4
dashboard = pd.read_csv(
    os.path.join(M4_OUT, "signal_dashboard.csv"),
    index_col=0)

# IC results from M4 (ICIR used as expected return proxy)
ic_results = pd.read_csv(
    os.path.join(M4_OUT, "ic_results.csv"),
    index_col=0)

T = len(factor_returns)
print(f"Factor returns:  {factor_returns.shape}")
print(f"Date range:      {factor_returns.index.min().date()} — "
      f"{factor_returns.index.max().date()}")

# ── Define investment universe: STRONG factors only ───────────
strong_factors = dashboard[
    dashboard['Rating'] == 'STRONG'].index.tolist()

# Confirm tier assignments
print(f"\nInvestment universe: {len(strong_factors)} STRONG factors")
print(f"\n{'─'*55}")
print(f"{'Factor':<16} {'Family':<15} {'Tier':<10} {'ICIR':>7}")
print(f"{'─'*55}")

for char in strong_factors:
    tier   = dashboard.loc[char, 'Tier']
    family = next(f for f, cs in CHAR_FAMILIES.items()
                  if char in cs)
    icir   = ic_results.loc[char, 'ICIR (ann)']
    print(f"  {char:<16} {family:<15} {tier:<10} {icir:>7.3f}")

# Factor returns for strong factors only
fr_strong = factor_returns[strong_factors].copy()

# Handle nulls: forward-fill then drop remaining
fr_strong = fr_strong.fillna(method='ffill').dropna()
print(f"\nClean factor returns: {fr_strong.shape}")

Loading data...
Factor returns:  (696, 41)
Date range:      1967-01-31 — 2024-12-31

Investment universe: 18 STRONG factors

───────────────────────────────────────────────────────
Factor           Family          Tier          ICIR
───────────────────────────────────────────────────────


KeyError: 'Tier'